# Lab 19: DDP and MLflow

**Module 19** | Read `notes/19-ddp-mlops.pdf` before running this notebook.

Train a small CNN on MNIST while logging parameters, metrics, and the model artifact to MLflow.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Experiment Tracking with MLflow

MLOps workflows need reproducible records of **what was trained, with which hyperparameters, and how well it performed**.
[MLflow Tracking](https://mlflow.org/docs/latest/tracking.html) logs params, metrics, and artifacts (including PyTorch models) to a local or remote store.

This lab trains a **small CNN on MNIST** for a few epochs and logs hyperparameters, per-epoch loss, test accuracy, and the model artifact.

The tracking URI comes from the environment (`MLFLOW_TRACKING_URI`), set by `runtime_env.configure_runtime()` to a local SQLite store under `mlruns/mlflow.db`.
Browse runs at http://127.0.0.1:5050 after launching the MLflow UI via `env.ps1` or `env.sh`.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **MLOps** | Practices for building, tracking, deploying, and monitoring machine learning systems reliably. |
| **MLflow Tracking** | Logs each training **run**: parameters, metrics over time, and artifacts (models, plots). Lets you compare experiments and reproduce results. |
| **Run** | One execution of a training script. MLflow assigns a unique run ID and stores everything under it. |
| **Parameters (params)** | Hyperparameters and config you choose before training (epochs, learning rate, batch size). Logged with `mlflow.log_param()`. |
| **Metrics** | Numbers measured during or after training (loss, accuracy). Logged with `mlflow.log_metric()`; can include a `step` for per-epoch curves. |
| **Artifacts** | Files saved with a run: model weights, configs, plots. `mlflow.pytorch.log_model()` saves the trained network. |
| **Tracking URI** | Where MLflow stores runs (local SQLite file, remote server, cloud bucket). Set via `MLFLOW_TRACKING_URI` environment variable. |
| **Distributed Data Parallel (DDP)** | PyTorch multi-GPU training: each GPU processes a batch, gradients are synchronized. Mentioned at the end of this lab for scaling beyond one device. |

Refer back to this table whenever a term appears in code or output.


### Model architecture

**What this cell does:** Loads project runtime settings and imports PyTorch.

**Why we do this:** `configure_runtime()` sets paths, device, and the MLflow tracking URI so later cells can log experiments without manual setup.


**What to look for in the output**

The cell should run without errors. No output is expected yet; it only prepares imports and runtime configuration.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import os

import torch
import torch.nn as nn


### Define SmallCNN

**What this cell does:** Defines a compact convolutional neural network for 28x28 MNIST digits and prints its architecture.

**Why we do this:** You need the model class before training. Two conv blocks with pooling shrink 28x28 down to 7x7 feature maps before the fully connected head.


**What to look for in the output**

You should see:
- `SmallCNN parameter count:` followed by a number (tens of thousands of parameters)
- A printed `Sequential` module listing Conv2d, ReLU, MaxPool2d, Linear, and Dropout layers


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # Conv block 1: 1 channel -> 16 channels, 28x28 spatial size
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # halve spatial size: 28 -> 14
            # Conv block 2: 16 -> 32 channels
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14 -> 7
            nn.Flatten(),
            # Fully connected head: 32*7*7 = 1568 features -> 128 -> 10 classes
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),  # regularization: randomly drop 25% of activations
            nn.Linear(128, 10),  # one logit per digit class (0-9)
        )

    def forward(self, x):
        return self.net(x)


model_preview = SmallCNN()
param_count = sum(p.numel() for p in model_preview.parameters())
print(f"SmallCNN parameter count: {param_count:,}")
print(model_preview)


### Load MNIST and build DataLoaders

**What this cell does:** Downloads MNIST, normalizes pixel values, creates train/test DataLoaders, and inspects one batch.

**Why we do this:** Normalization (mean 0.1307, std 0.3081) matches standard MNIST preprocessing and helps training converge. Checking batch shapes confirms data pipeline correctness.


**What to look for in the output**

You should see:
- Training device (cpu or cuda)
- Train samples: 60,000 and test samples: 10,000
- Batch image shape (128, 1, 28, 28)
- Batch label shape (128,)
- Sample labels 0-9 and label dtype int64


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Standard MNIST normalization constants
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

data_root = ROOT / "datasets" / "mnist"
train_ds = datasets.MNIST(str(data_root), train=True, download=True, transform=transform)
test_ds = datasets.MNIST(str(data_root), train=False, download=True, transform=transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

# Peek at one batch to verify shapes
xb, yb = next(iter(train_loader))
print(f"Training device: {device}")
print(f"Train samples: {len(train_ds):,}, test samples: {len(test_ds):,}")
print(f"Batch image shape: {tuple(xb.shape)}  (batch, channels, height, width)")
print(f"Batch label shape: {tuple(yb.shape)}")
print(f"Sample batch labels (first 16): {yb[:16].tolist()}")
print(f"Label dtype: {yb.dtype}, value range: {int(yb.min())} to {int(yb.max())}")

epochs = 2
lr = 1e-3


### Train, evaluate, and log to MLflow

**What this cell does:** Trains SmallCNN for 2 epochs, evaluates on the test set, and logs everything to MLflow inside a single run.

**Why we do this:** MLflow creates an audit trail. **What MLflow tracks here:**
- **Params:** epochs, learning rate, batch size, device, architecture name
- **Metrics:** per-epoch train_loss and train_accuracy (with step index), final test_accuracy
- **Artifacts:** the full PyTorch model saved under `mnist_cnn`

You can open the MLflow UI later to compare runs, download the model, or share results with teammates.


**What to look for in the output**

You should see:
- MLflow tracking URI printed
- TRAINING block with 2 epoch lines showing falling loss and rising accuracy
- TEST SET block with test accuracy above 95% (typically 97%+ after 2 epochs)
- Sample predictions table for 10 test images with true/predicted digits and top-3 probabilities
- MLFLOW RUN INFO with run ID, run name, experiment ID, artifact URI, and UI URL


In [ ]:
import mlflow
import mlflow.pytorch

# Connect to the tracking store configured by runtime_env
tracking_uri = os.environ["MLFLOW_TRACKING_URI"]
mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

# Default experiment may use mlflow-artifacts:/ (needs an HTTP server). Use a file:// store instead.
exp_name = "lab19_mnist_cnn"
artifacts_dir = ROOT / "mlruns" / "experiments" / exp_name
artifacts_dir.mkdir(parents=True, exist_ok=True)
artifact_uri = "file:///" + artifacts_dir.resolve().as_posix()
try:
    mlflow.create_experiment(exp_name, artifact_location=artifact_uri)
except Exception:
    pass  # experiment already exists from a previous run
mlflow.set_experiment(exp_name)

# Everything inside this block is one MLflow "run"
with mlflow.start_run(run_name="lab19_mnist_cnn") as run:
    # Log hyperparameters (fixed config for this experiment)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("learning_rate", lr)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("device", str(device))
    mlflow.log_param("architecture", "SmallCNN")

    model = SmallCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    print("=" * 50)
    print("TRAINING")
    print("=" * 50)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        # Log metrics with step=epoch so MLflow plots them as a time series
        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        mlflow.log_metric("train_accuracy", epoch_acc, step=epoch)
        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"loss={epoch_loss:.4f}, train accuracy={epoch_acc:.4f}"
        )

    model.eval()
    test_correct = 0
    test_total = 0
    all_logits: list[torch.Tensor] = []
    all_labels: list[torch.Tensor] = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            test_correct += (preds == yb).sum().item()
            test_total += yb.size(0)
            all_logits.append(logits.cpu())
            all_labels.append(yb.cpu())

    test_acc = test_correct / test_total
    mlflow.log_metric("test_accuracy", test_acc)
    # Log on CPU with pickle format (portable; avoids pt2 tracing/signature requirements)
    sample_input, _ = next(iter(test_loader))
    input_example = sample_input[:1].cpu().numpy()
    mlflow.pytorch.log_model(
        model.cpu(),
        artifact_path="mnist_cnn",
        input_example=input_example,
        serialization_format="pickle",
    )

    print()
    print("=" * 50)
    print("TEST SET")
    print("=" * 50)
    print(f"Test accuracy: {test_acc:.4f} ({test_correct}/{test_total} correct)")

    logits_cat = torch.cat(all_logits, dim=0)
    labels_cat = torch.cat(all_labels, dim=0)
    probs = torch.softmax(logits_cat[:10], dim=1)

    print()
    print("Sample predictions (first 10 test images):")
    print(f"{'Idx':>4} | {'True':>4} | {'Pred':>4} | Top-3 probabilities (digit: prob)")
    print("-" * 60)
    for i in range(10):
        true_digit = int(labels_cat[i])
        pred_digit = int(logits_cat[i].argmax())
        top3 = probs[i].topk(3)
        top3_str = ", ".join(
            f"{int(d)}:{p:.3f}" for d, p in zip(top3.indices, top3.values)
        )
        print(f"{i:>4} | {true_digit:>4} | {pred_digit:>4} | {top3_str}")

    print()
    print("=" * 50)
    print("MLFLOW RUN INFO")
    print("=" * 50)
    print(f"Run ID: {run.info.run_id}")
    print(f"Run name: {run.info.run_name}")
    print(f"Experiment ID: {run.info.experiment_id}")
    print(f"Artifact URI: {mlflow.get_artifact_uri('mnist_cnn')}")
    mlflow_port = os.environ.get("MLFLOW_UI_PORT", "5050")
    print(f"Open MLflow UI at http://127.0.0.1:{mlflow_port} to inspect params, metrics, and model.")


## Optional: Distributed Data Parallel (DDP)

For multi-GPU training, PyTorch **DistributedDataParallel** wraps the model and synchronizes gradients across processes launched with `torchrun`:

```bash
torchrun --nproc_per_node=2 train_script.py
```

Each process runs on one GPU; only rank 0 should call `mlflow.log_*` to avoid duplicate runs.
This lab uses single-process training so it runs on any machine; the MLflow logging pattern is the same in DDP, only the launch command differs.

**Deliverable:** note the printed **run ID** and confirm the run appears in the MLflow UI with `test_accuracy` and a `mnist_cnn` model artifact.
